In [ ]:
!pip install nba_api

In [ ]:
import pandas as pd
import requests
import time

from nba_api.stats.endpoints import leaguedashplayerstats
import requests
%pip install scikit-learn

In [ ]:
response = leaguedashplayerstats.LeagueDashPlayerStats(
    season="2025-26",
    season_type_all_star="Regular Season",
    per_mode_detailed="PerGame"
)

stats = response.get_data_frames()[0]

performance = stats[
    [
        "PLAYER_ID",
        "PLAYER_NAME",
        "TEAM_ABBREVIATION",
        "AGE",
        "GP",
        "MIN",
        "PTS",
        "REB",
        "AST"
    ]
]

performance.head()

In [ ]:
performance.to_csv("nba_performance.csv", index=False)

In [ ]:
import os
from getpass import getpass

SPORTRADAR_API_KEY = os.getenv("SPORTRADAR_API_KEY")
if not SPORTRADAR_API_KEY:
    SPORTRADAR_API_KEY = getpass("Enter your Sportradar API key: ")


In [ ]:
hierarchy_url = (
    "https://api.sportradar.com/nba/trial/v8/en/"
    "league/hierarchy.json"
)

response = requests.get(
    hierarchy_url,
    headers={"x-api-key": SPORTRADAR_API_KEY}
)

print(response.status_code)

In [ ]:
hierarchy = response.json()

print(hierarchy.keys())

In [ ]:
teams = []

for conference in hierarchy["conferences"]:
    for division in conference["divisions"]:
        for team in division["teams"]:
            teams.append({
                "team_id": team["id"],
                "team_name": team["name"],
                "team_market": team["market"],
                "team_abbreviation": team["alias"]
            })

teams_df = pd.DataFrame(teams)

teams_df.head()

In [ ]:
test_team_id = teams_df.loc[0, "team_id"]

team_url = (
    f"https://api.sportradar.com/nba/trial/v8/en/"
    f"teams/{test_team_id}/profile.json"
)

response = requests.get(
    team_url,
    headers={"x-api-key": SPORTRADAR_API_KEY}
)

print(response.status_code)

team_profile = response.json()
print(team_profile.keys())

In [ ]:
players = team_profile.get("players", [])

print("Number of players:", len(players))
print(players[0].keys())

In [ ]:
salary_rows = []

for player in players:
    salary_rows.append({
        "NBA_PLAYER_ID": player.get("reference"),
        "PLAYER_NAME": player.get("full_name"),
        "TEAM": team_profile.get("alias"),
        "POSITION": player.get("primary_position"),
        "EXPERIENCE": player.get("experience"),
        "SALARY": player.get("salary"),
        "STATUS": player.get("status")
    })

kings_salaries = pd.DataFrame(salary_rows)

kings_salaries

In [ ]:
print(kings_salaries[[
    "PLAYER_NAME",
    "SALARY"
]])

In [ ]:
print("Players with salary data:", kings_salaries["SALARY"].notna().sum())
print("Players without salary data:", kings_salaries["SALARY"].isna().sum())

In [ ]:
all_salary_rows = []

for index, team in teams_df.iterrows():
    team_id = team["team_id"]
    team_abbreviation = team["team_abbreviation"]

    team_url = (
        "https://api.sportradar.com/nba/trial/v8/en/"
        f"teams/{team_id}/profile.json"
    )

    response = requests.get(
        team_url,
        headers={"x-api-key": SPORTRADAR_API_KEY}
    )

    if response.status_code == 200:
        team_data = response.json()
        players = team_data.get("players", [])

        for player in players:
            all_salary_rows.append({
                "NBA_PLAYER_ID": player.get("reference"),
                "PLAYER_NAME": player.get("full_name"),
                "TEAM": team_abbreviation,
                "POSITION": player.get("primary_position"),
                "EXPERIENCE": player.get("experience"),
                "SALARY": player.get("salary"),
                "STATUS": player.get("status")
            })

        print(team_abbreviation, "- collected", len(players), "players")

    else:
        print(
            team_abbreviation,
            "- request failed:",
            response.status_code
        )

    # Avoid sending requests too quickly
    time.sleep(1)

In [ ]:
salaries = pd.DataFrame(all_salary_rows)

print("Rows before removing duplicates:", len(salaries))

In [ ]:
salaries = salaries.drop_duplicates(
    subset=["NBA_PLAYER_ID", "PLAYER_NAME", "TEAM"],
    keep="last"
).reset_index(drop=True)

print("Rows after removing duplicates:", len(salaries))
print("Teams collected:", salaries["TEAM"].nunique())

In [ ]:
print(sorted(salaries["TEAM"].unique()))

In [ ]:
time.sleep(10)

phx_team = teams_df[
    teams_df["team_abbreviation"] == "PHX"
].iloc[0]

phx_url = (
    "https://api.sportradar.com/nba/trial/v8/en/"
    f"teams/{phx_team['team_id']}/profile.json"
)

response = requests.get(
    phx_url,
    headers={"x-api-key": SPORTRADAR_API_KEY}
)

print(response.status_code)

In [ ]:
phx_data = response.json()

for player in phx_data.get("players", []):
    all_salary_rows.append({
        "NBA_PLAYER_ID": player.get("reference"),
        "PLAYER_NAME": player.get("full_name"),
        "TEAM": "PHX",
        "POSITION": player.get("primary_position"),
        "EXPERIENCE": player.get("experience"),
        "SALARY": player.get("salary"),
        "STATUS": player.get("status")
    })

In [ ]:
salaries = pd.DataFrame(all_salary_rows)

salaries = salaries.drop_duplicates(
    subset=["NBA_PLAYER_ID", "PLAYER_NAME", "TEAM"],
    keep="last"
).reset_index(drop=True)

print("Teams collected:", salaries["TEAM"].nunique())
print(sorted(salaries["TEAM"].unique()))

In [ ]:
salaries.to_csv("nba_salaries.csv", index=False)

In [ ]:
nba_response = leaguedashplayerstats.LeagueDashPlayerStats(
    season="2025-26",
    season_type_all_star="Regular Season",
    per_mode_detailed="PerGame",
    timeout=60
)

performance = nba_response.get_data_frames()[0]

performance.head()

In [ ]:
missing_teams = teams_df[
    ~teams_df["team_abbreviation"].isin(salaries["TEAM"])
]

missing_teams

In [ ]:
sac_team = teams_df[
    teams_df["team_abbreviation"] == "SAC"
].iloc[0]

time.sleep(10)

sac_url = (
    "https://api.sportradar.com/nba/trial/v8/en/"
    f"teams/{sac_team['team_id']}/profile.json"
)

response = requests.get(
    sac_url,
    headers={"x-api-key": SPORTRADAR_API_KEY}
)

print(response.status_code)

In [ ]:
sac_data = response.json()

for player in sac_data.get("players", []):
    all_salary_rows.append({
        "NBA_PLAYER_ID": player.get("reference"),
        "PLAYER_NAME": player.get("full_name"),
        "TEAM": "SAC",
        "POSITION": player.get("primary_position"),
        "EXPERIENCE": player.get("experience"),
        "SALARY": player.get("salary"),
        "STATUS": player.get("status")
    })

print("SAC - collected", len(sac_data.get("players", [])), "players")

In [ ]:
salaries = pd.DataFrame(all_salary_rows)

salaries = salaries.drop_duplicates(
    subset=["NBA_PLAYER_ID", "PLAYER_NAME", "TEAM"],
    keep="last"
).reset_index(drop=True)

print("Teams collected:", salaries["TEAM"].nunique())
print(sorted(salaries["TEAM"].unique()))

salaries.to_csv("nba_salaries.csv", index=False)

In [ ]:
performance = performance[
    [
        "PLAYER_ID",
        "PLAYER_NAME",
        "TEAM_ABBREVIATION",
        "AGE",
        "GP",
        "MIN",
        "PTS",
        "REB",
        "AST"
    ]
].copy()

performance.head()

In [ ]:
performance["PLAYER_ID"] = pd.to_numeric(
    performance["PLAYER_ID"],
    errors="coerce"
).astype("Int64")

salaries["NBA_PLAYER_ID"] = pd.to_numeric(
    salaries["NBA_PLAYER_ID"],
    errors="coerce"
).astype("Int64")

In [ ]:
final_data = performance.merge(
    salaries,
    left_on="PLAYER_ID",
    right_on="NBA_PLAYER_ID",
    how="inner"
)

final_data.head()

In [ ]:
print("Performance records:", len(performance))
print("Salary records:", len(salaries))
print("Successfully matched records:", len(final_data))
print("Players with salary:", final_data["SALARY"].notna().sum())
print("Players without salary:", final_data["SALARY"].isna().sum())

In [ ]:
analysis_data = final_data[
    (final_data["GP"] >= 20) &
    (final_data["SALARY"].notna())
].copy()

print("Players available for analysis:", len(analysis_data))

In [ ]:
print(
    "Duplicate player IDs:",
    analysis_data["PLAYER_ID"].duplicated().sum()
)

In [ ]:
analysis_data = analysis_data[
    [
        "PLAYER_ID",
        "PLAYER_NAME_x",
        "TEAM_ABBREVIATION",
        "AGE",
        "GP",
        "MIN",
        "PTS",
        "REB",
        "AST",
        "POSITION",
        "EXPERIENCE",
        "SALARY"
    ]
].copy()

analysis_data = analysis_data.rename(
    columns={
        "PLAYER_NAME_x": "PLAYER_NAME",
        "TEAM_ABBREVIATION": "PERFORMANCE_TEAM"
    }
)

analysis_data["SALARY_MILLIONS"] = (
    analysis_data["SALARY"] / 1_000_000
)

analysis_data.head()

In [ ]:
variables = [
    "SALARY_MILLIONS",
    "PTS",
    "AST",
    "REB",
    "MIN",
    "GP"
]

correlations = analysis_data[variables].corr()

correlations

In [ ]:
salary_correlations = correlations[
    "SALARY_MILLIONS"
].drop("SALARY_MILLIONS").sort_values(ascending=False)

salary_correlations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(9, 5))

sns.barplot(
    x=salary_correlations.values,
    y=salary_correlations.index
)

plt.title("Correlation Between NBA Performance and Salary")
plt.xlabel("Correlation with Annual Salary")
plt.ylabel("Performance Statistic")
plt.xlim(-0.1, 1)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

sns.regplot(
    data=analysis_data,
    x="PTS",
    y="SALARY_MILLIONS",
    scatter_kws={"alpha": 0.6},
    line_kws={"color": "red"}
)

plt.title("NBA Points Per Game and Annual Salary")
plt.xlabel("Points Per Game")
plt.ylabel("Annual Salary ($ Millions)")

plt.tight_layout()
plt.show()

In [ ]:
summary = analysis_data[
    ["SALARY_MILLIONS", "PTS", "AST", "REB", "MIN", "GP"]
].describe().round(2)

summary

In [ ]:
performance_variables = ["PTS", "AST", "REB", "MIN", "GP"]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, variable in enumerate(performance_variables):
    sns.regplot(
        data=analysis_data,
        x=variable,
        y="SALARY_MILLIONS",
        ax=axes[i],
        scatter_kws={"alpha": 0.5},
        line_kws={"color": "red"}
    )

    axes[i].set_title(f"{variable} and Annual Salary")
    axes[i].set_xlabel(variable)
    axes[i].set_ylabel("Salary ($ Millions)")

axes[5].axis("off")

plt.suptitle(
    "Relationships Between NBA Performance and Salary",
    fontsize=16
)

plt.tight_layout()

plt.savefig(
    "all_performance_salary_relationships.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=analysis_data,
    x="POSITION",
    y="SALARY_MILLIONS"
)

plt.title("NBA Annual Salary by Player Position")
plt.xlabel("Player Position")
plt.ylabel("Annual Salary ($ Millions)")

plt.tight_layout()
plt.show()

In [ ]:
position_summary = analysis_data.groupby("POSITION").agg(
    PLAYER_COUNT=("PLAYER_ID", "count"),
    AVERAGE_SALARY=("SALARY_MILLIONS", "mean"),
    MEDIAN_SALARY=("SALARY_MILLIONS", "median"),
    AVERAGE_POINTS=("PTS", "mean")
).round(2)

position_summary

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

predictors = ["PTS", "AST", "REB", "MIN", "GP"]

X = analysis_data[predictors]
y = analysis_data[["SALARY_MILLIONS"]]

scaler_x = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_x.fit_transform(X)
y_scaled = scaler_y.fit_transform(y).ravel()

model = LinearRegression()
model.fit(X_scaled, y_scaled)

regression_results = pd.DataFrame({
    "STATISTIC": predictors,
    "STANDARDIZED_COEFFICIENT": model.coef_
}).sort_values(
    "STANDARDIZED_COEFFICIENT",
    ascending=False
)

print(regression_results)
print("R-squared:", round(model.score(X_scaled, y_scaled), 3))